# 🚔 Crime Pattern Prediction System
**Project:** End-to-end Data Science & Machine Learning project for predicting crime patterns.

**Author:** Student Project | **Tech Stack:** Python, Pandas, scikit-learn, Matplotlib, Seaborn

## Objectives
1. Perform Exploratory Data Analysis (EDA) on crime data
2. Identify spatial hotspots using clustering
3. Train multiple ML classifiers to predict high-risk incidents
4. Compare model performance and select the best one
5. Visualize temporal & spatial patterns

## 1. Imports & Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.cluster import KMeans
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                              confusion_matrix, classification_report, roc_auc_score, roc_curve)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize']=(10,5)

## 2. Load Dataset

In [ ]:
df = pd.read_csv('../data/crime_data.csv', parse_dates=['timestamp','date'])
print('Shape:', df.shape)
df.head()

In [ ]:
df.info()
df.describe()

## 3. Exploratory Data Analysis

In [ ]:
df['crime_type'].value_counts().plot(kind='bar', color=sns.color_palette('magma',8))
plt.title('Crime Type Distribution'); plt.ylabel('Count'); plt.show()

In [ ]:
pivot = df.pivot_table(index='day_of_week', columns='hour', values='severity', aggfunc='count').fillna(0)
order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
pivot = pivot.reindex(order)
plt.figure(figsize=(12,4))
sns.heatmap(pivot, cmap='YlOrRd'); plt.title('Crime Heatmap: Day × Hour'); plt.show()

In [ ]:
daily = df.groupby('date').size()
plt.figure(figsize=(12,4))
daily.plot(alpha=0.4, label='Daily')
daily.rolling(7).mean().plot(color='red', label='7-day MA')
plt.title('Daily Crime Trend'); plt.legend(); plt.show()

## 4. Hotspot Detection with KMeans

In [ ]:
coords = df[['latitude','longitude']].values
km = KMeans(n_clusters=6, random_state=42, n_init=10)
df['hotspot'] = km.fit_predict(coords)
plt.figure(figsize=(9,7))
for c in range(6):
    s = df[df['hotspot']==c]
    plt.scatter(s['longitude'], s['latitude'], s=8, alpha=0.5, label=f'Zone {c+1}')
plt.scatter(km.cluster_centers_[:,1], km.cluster_centers_[:,0], marker='X', s=200, c='black')
plt.title('Crime Hotspots'); plt.legend(fontsize=8); plt.show()

## 5. Feature Engineering & Preprocessing

In [ ]:
X = df[['hour','month','is_weekend','severity','temperature_c',
        'population_density','prev_incidents_30d',
        'district','crime_type','weather','day_of_week']].copy()
y = df['high_risk']
for col in ['district','crime_type','weather','day_of_week']:
    X[col] = LabelEncoder().fit_transform(X[col])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print('Train:', X_train.shape, 'Test:', X_test.shape)

## 6. Train Multiple Models

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Decision Tree': DecisionTreeClassifier(max_depth=10, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=200, max_depth=15, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=150, max_depth=5, random_state=42),
}
scaler = StandardScaler(); Xtr_s = scaler.fit_transform(X_train); Xte_s = scaler.transform(X_test)
results = {}
for name, m in models.items():
    Xtr = Xtr_s if name=='Logistic Regression' else X_train
    Xte = Xte_s if name=='Logistic Regression' else X_test
    m.fit(Xtr, y_train); p = m.predict(Xte); pr = m.predict_proba(Xte)[:,1]
    results[name] = {'accuracy':accuracy_score(y_test,p),'precision':precision_score(y_test,p),
                     'recall':recall_score(y_test,p),'f1':f1_score(y_test,p),'roc_auc':roc_auc_score(y_test,pr)}
pd.DataFrame(results).T.round(4)

## 7. Model Comparison & ROC

In [ ]:
pd.DataFrame(results).T.plot(kind='bar', figsize=(10,5), colormap='viridis')
plt.title('Model Comparison'); plt.ylim(0,1.05); plt.show()

In [ ]:
plt.figure(figsize=(7,5))
for name, m in models.items():
    Xte = Xte_s if name=='Logistic Regression' else X_test
    fpr,tpr,_ = roc_curve(y_test, m.predict_proba(Xte)[:,1])
    plt.plot(fpr,tpr,label=f"{name} (AUC={results[name]['roc_auc']:.3f})")
plt.plot([0,1],[0,1],'k--',alpha=0.5); plt.legend(); plt.title('ROC Curves'); plt.show()

## 8. Conclusion
- **Random Forest / Gradient Boosting** typically perform best on this task.
- KMeans identifies actionable spatial **hotspots** for police patrols.
- Strong **temporal patterns** show late-night & weekend spikes.
- The system can guide **resource allocation** & **preventive policing**.